## Architecture Redesign: Early Fusion

### Previous Design (Late Fusion - Suboptimal)
- Vertex features → concatenated only at final classification layer (EdgeAnomalyHead)
- GNN learned node embeddings without entity-type information
- Result: Limited discriminative power, poor learned representations

### New Design (Early Fusion - Optimal)
- Vertex features → fused with memory embeddings **before** graph convolution
- GNN learns entity-aware node embeddings from temporal context
- Vertex features also used at final classification for multi-scale fusion
- Result: Richer node representations, better edge scoring

### Expected Improvements
- ✅ Higher PR-AUC and ROC-AUC scores (better discrimination)
- ✅ More stable threshold calibration
- ✅ Entity-type information shapes temporal aggregation behavior
- ✅ Better convergence during training

# TGNMemory Anomaly Detection

This notebook keeps the same preprocessing flow and TGNMemory backbone from the classification notebook, but turns the task into anomaly scoring. The model still learns from the `Is Laundering` signal, yet the output is treated as an anomaly logit and evaluated with validation-calibrated thresholds.

In [2]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
from time import perf_counter
from typing import Literal
import random

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from torch.nn import Linear
from torch.optim import AdamW
from torch_geometric.data import TemporalData
from torch_geometric.loader import TemporalDataLoader
from torch_geometric.nn import TGNMemory, TransformerConv
from torch_geometric.nn.models.tgn import (
    IdentityMessage,
    LastNeighborLoader,
    MeanAggregator,
)
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

/Users/darrellcr/Devs/apple_aiml/c1/tabular/python/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset_path = Path('../dataset')
artifact_dir = Path('./preprocessors')
checkpoint_dir = Path('./checkpoints/tgn_anomaly')
log_dir = Path('./logs/tensorboard')

artifact_dir.mkdir(parents=True, exist_ok=True)
checkpoint_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

accounts_path = dataset_path / 'HI-Small_accounts.csv'
transactions_path = dataset_path / 'HI-Small_Trans.csv'
device = torch.device('cpu')

print(f'{accounts_path=}')
print(f'{transactions_path=}')
print(f'{device=}')

accounts_path=PosixPath('../dataset/HI-Small_accounts.csv')
transactions_path=PosixPath('../dataset/HI-Small_Trans.csv')
device=device(type='cpu')


In [4]:
accounts = pd.read_csv(accounts_path)
accounts['Bank Name'] = accounts['Bank Name'].str.split(' #').str[0]
accounts['Entity Name'] = accounts['Entity Name'].str.split(' #').str[0]
print(f'{accounts.shape=}')
accounts.head()

accounts.shape=(518581, 5)


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Portugal Bank,331579,80B779D80,80062E240,Sole Proprietorship
1,Canada Bank,210,809D86900,800C998A0,Corporation
2,UK Bank,21884,80812BE00,800C47F50,Partnership
3,Germany Bank,32742,81047F300,80096F0B0,Corporation
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation


In [5]:
transactions = pd.read_csv(transactions_path)
transactions = transactions.rename(columns={'Account': 'From Account', 'Account.1': 'To Account'})
transactions['Timestamp'] = pd.to_datetime(transactions['Timestamp'])
print(f'{transactions.shape=}')
transactions.head()

transactions.shape=(5078345, 11)


,Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022-09-01 00:20:00,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022-09-01 00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022-09-01 00:00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022-09-01 00:02:00,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022-09-01 00:06:00,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


## Preprocessing

The preprocessing matches the original notebook: `Entity Name` is one-hot encoded for vertices, while transaction edges use categorical one-hot features plus robust-scaled numeric features.

In [6]:
accounts_iterator = accounts.loc[:, ['Bank ID', 'Account Number']].itertuples()
account_to_idx = {(bank_id, account_number): idx for idx, (_, bank_id, account_number) in enumerate(accounts_iterator)}

# --- Node feature engineering: per-account aggregates ---
transactions['Timestamp'] = pd.to_datetime(transactions['Timestamp'])

# Outgoing (sent) statistics
outgoing = transactions.groupby(['From Bank', 'From Account']).agg(
    outgoing_mean_amount=('Amount Paid', 'mean'),
    outgoing_std_amount=('Amount Paid', 'std'),
    count_outgoing=('Amount Paid', 'count'),
    mean_outgoing_time_gap=('Timestamp', lambda x: x.sort_values().diff().dt.total_seconds().mean()),
).reset_index().rename(columns={'From Bank': 'Bank ID', 'From Account': 'Account Number'})

# Incoming (received) statistics
incoming = transactions.groupby(['To Bank', 'To Account']).agg(
    incoming_mean_amount=('Amount Received', 'mean'),
    incoming_std_amount=('Amount Received', 'std'),
    count_incoming=('Amount Received', 'count'),
    mean_incoming_time_gap=('Timestamp', lambda x: x.sort_values().diff().dt.total_seconds().mean()),
).reset_index().rename(columns={'To Bank': 'Bank ID', 'To Account': 'Account Number'})

# Merge aggregates into accounts dataframe
accounts_feats = accounts.merge(outgoing, on=['Bank ID', 'Account Number'], how='left').merge(incoming, on=['Bank ID', 'Account Number'], how='left')

# Fill missing numeric values with sensible defaults
for col in ['outgoing_mean_amount', 'outgoing_std_amount', 'count_outgoing', 'mean_outgoing_time_gap',
            'incoming_mean_amount', 'incoming_std_amount', 'count_incoming', 'mean_incoming_time_gap']:
    if col in accounts_feats.columns:
        accounts_feats[col] = accounts_feats[col].fillna(0.0)

# Mean time gap across all transactions involving the account (seconds)
accounts_feats['mean_time_gap'] = accounts_feats[['mean_outgoing_time_gap', 'mean_incoming_time_gap']].replace(0, np.nan).mean(axis=1).fillna(0.0)

# Numeric columns to include in vertex features
numeric_cols = ['incoming_mean_amount', 'incoming_std_amount', 'outgoing_mean_amount', 'outgoing_std_amount',
                'count_incoming', 'count_outgoing', 'mean_time_gap']

vertex_transformer = ColumnTransformer([
    ('categorical_features', OneHotEncoder(sparse_output=False, drop='first'), ['Entity Name']),
    ('numeric', RobustScaler(), numeric_cols),
], remainder='drop')

# Build vertex tensor from enriched accounts dataframe
vertices_np = vertex_transformer.fit_transform(accounts_feats)
vertices = torch.tensor(vertices_np, dtype=torch.float32)
joblib.dump(vertex_transformer, artifact_dir / 'vertex_preprocessor.joblib')
print(f'{vertices.shape=}')

vertices.shape=torch.Size([518581, 12])


In [7]:
accounts_feats

,Bank Name,Bank ID,Account Number,Entity ID,Entity Name,outgoing_mean_amount,outgoing_std_amount,count_outgoing,mean_outgoing_time_gap,incoming_mean_amount,incoming_std_amount,count_incoming,mean_incoming_time_gap,mean_time_gap
0,Portugal Bank,331579,80B779D80,80062E240,Sole Proprietorship,1.657800e+02,0.000000e+00,2.0,575280.000000,0.000000,0.000000,0.0,0.000000,575280.000000
1,Canada Bank,210,809D86900,800C998A0,Corporation,0.000000e+00,0.000000e+00,0.0,0.000000,346.080000,0.000000,2.0,598260.000000,598260.000000
2,UK Bank,21884,80812BE00,800C47F50,Partnership,1.487449e+06,1.443522e+06,4.0,400.000000,0.000000,0.000000,0.0,0.000000,400.000000
3,Germany Bank,32742,81047F300,80096F0B0,Corporation,7.811400e+02,0.000000e+00,2.0,598560.000000,0.000000,0.000000,0.0,0.000000,598560.000000
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation,6.411000e+01,0.000000e+00,1.0,0.000000,534.776667,407.609290,3.0,345300.000000,345300.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
518576,France Bank,3881,807886B70,80062D160,Sole Proprietorship,1.534579e+03,3.166012e+03,16.0,46968.000000,1580.293043,2680.564530,23.0,38293.636364,42630.818182
518577,National Bank of Topeka,333423,81314C870,800F40230,Sole Proprietorship,5.123500e+02,0.000000e+00,1.0,0.000000,512.350000,0.000000,1.0,0.000000,0.000000
518578,Plandor Trust Bank,1467,804ED2270,800CAC4C0,Sole Proprietorship,2.792825e+02,8.184077e+02,53.0,16293.461538,640.266667,1056.600282,3.0,129480.000000,72886.730769
518579,Sappo Bancorp,2843,801727270,800EFCB40,Sole Proprietorship,1.368586e+04,7.611011e+03,5.0,167835.000000,2216.462500,2597.207561,8.0,103954.285714,135894.642857


In [6]:
# Edge feature engineering: cross-currency, cross-bank, hour, and time-since-last
# Note: relies on transactions['Timestamp'] being datetime from earlier preprocessing

categorical_features = ['Payment Format', 'Receiving Currency', 'Payment Currency']

# Cross flags
transactions['is_cross_currency'] = (transactions['Receiving Currency'] != transactions['Payment Currency']).astype(int)
transactions['is_cross_bank'] = (transactions['From Bank'] != transactions['To Bank']).astype(int)

# Hour of day
transactions['hour_of_day'] = pd.to_datetime(transactions['Timestamp']).dt.hour.astype(int)

# Ensure ordering
transactions = transactions.sort_values('Timestamp').reset_index(drop=True)

# Time since last outgoing from source (seconds)
transactions['time_since_src_last_txn'] = transactions.groupby(['From Bank', 'From Account'])['Timestamp'].diff().dt.total_seconds().fillna(0.0)
# Time since last incoming to destination (seconds)
transactions['time_since_dst_last_txn'] = transactions.groupby(['To Bank', 'To Account'])['Timestamp'].diff().dt.total_seconds().fillna(0.0)

# Include new engineered numeric features alongside existing amount columns
numerical_features = ['Amount Received', 'Amount Paid', 'is_cross_currency', 'is_cross_bank', 'hour_of_day',
                      'time_since_src_last_txn', 'time_since_dst_last_txn']

edge_transformer = ColumnTransformer([
    ('categorical', OneHotEncoder(sparse_output=False), categorical_features),
    ('numerical', RobustScaler(), numerical_features),
], remainder='passthrough', verbose_feature_names_out=False)
edge_transformer.set_output(transform='pandas')

preprocessed_transactions = edge_transformer.fit_transform(transactions)
# Drop columns that shouldn't be part of the model input
to_remove = ['From Bank', 'From Account', 'To Account', 'To Bank', 'Is Laundering', 'Timestamp']
preprocessed_transactions = preprocessed_transactions.drop(columns=[c for c in to_remove if c in preprocessed_transactions.columns])
edge_features = torch.tensor(preprocessed_transactions.values, dtype=torch.float32)
joblib.dump(edge_transformer, artifact_dir / 'edge_preprocessor.joblib')
print(f'{edge_features.shape=}')
preprocessed_transactions.head()

edge_features.shape=torch.Size([5078345, 44])


,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Receiving Currency_Australian Dollar,Receiving Currency_Bitcoin,Receiving Currency_Brazil Real,...,Payment Currency_US Dollar,Payment Currency_Yen,Payment Currency_Yuan,Amount Received,Amount Paid,is_cross_currency,is_cross_bank,hour_of_day,time_since_src_last_txn,time_since_dst_last_txn
0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,-0.112093,-0.112842,0.0,-1.0,-0.769231,-0.027842,-0.02019
1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.206070,0.206621,0.0,-1.0,-0.769231,-0.027842,-0.02019
2,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,-0.107994,-0.108727,0.0,0.0,-0.769231,-0.027842,-0.02019
3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.030021,1.033942,0.0,0.0,-0.769231,-0.027842,-0.02019
4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,-0.115157,-0.115919,0.0,-1.0,-0.769231,-0.027842,-0.02019


In [7]:
# Build edge index and TemporalData
source_keys = list(zip(transactions['From Bank'], transactions['From Account']))
destination_keys = list(zip(transactions['To Bank'], transactions['To Account']))
transactions['source'] = pd.Series(source_keys).map(account_to_idx)
transactions['destination'] = pd.Series(destination_keys).map(account_to_idx)
edge_index = torch.tensor(transactions[['source', 'destination']].values.T, dtype=torch.long)

edge_labels = torch.tensor(transactions['Is Laundering'].values, dtype=torch.float32)
unix_timestamp = transactions['Timestamp'].astype('int64') // 10**9

# TemporalData expects src, dst, t, msg, y
data = TemporalData(
    src=edge_index[0],
    dst=edge_index[1],
    t=torch.tensor(unix_timestamp.values, dtype=torch.long),
    msg=edge_features,
    y=edge_labels,
).to(device)

train_data, val_data, test_data = data.train_val_test_split(val_ratio=0.15, test_ratio=0.15)
train_loader = TemporalDataLoader(train_data, batch_size=512)
val_loader = TemporalDataLoader(val_data, batch_size=512)
test_loader = TemporalDataLoader(test_data, batch_size=512)
neighbor_loader = LastNeighborLoader(data.num_nodes, size=15, device=device)

print(f'{data.num_events=}, {data.num_nodes=}, msg_dim={data.msg.size(-1)}')

data.num_events=5078345, data.num_nodes=518581, msg_dim=44


In [8]:
class GraphAttentionEmbedding(torch.nn.Module):
    def __init__(self, in_channels: int, out_channels: int, msg_dim: int, time_enc, vertex_dim: int | None = None, num_heads: int = 4) -> None:
        super().__init__()
        self.time_enc = time_enc
        self.vertex_dim = vertex_dim
        self.output_dim = out_channels
        edge_dim = msg_dim + time_enc.out_channels

        # Early fusion: project vertex features and fuse with memory embeddings.
        if vertex_dim is not None:
            self.vertex_proj = Linear(vertex_dim, in_channels)
        else:
            self.vertex_proj = None

        self.conv = TransformerConv(
            in_channels,
            out_channels,
            heads=num_heads,
            concat=False,
            dropout=0.1,
            edge_dim=edge_dim,
        )

    def forward(self, x, last_update, edge_index, t, msg, vertex_features=None):
        # Early fusion: incorporate vertex features into node embeddings before convolution.
        if vertex_features is not None and self.vertex_proj is not None:
            vertex_proj = self.vertex_proj(vertex_features)
            x = x + vertex_proj

        rel_t = last_update[edge_index[0]] - t
        rel_t_enc = self.time_enc(rel_t.to(x.dtype))
        edge_attr = torch.cat([rel_t_enc, msg], dim=-1)
        return self.conv(x, edge_index, edge_attr)


class EdgeAnomalyHead(torch.nn.Module):
    def __init__(self, node_dim: int, edge_dim: int, vertex_dim: int = 0) -> None:
        super().__init__()
        self.node_dim = node_dim
        self.vertex_dim = vertex_dim
        head_input_dim = node_dim * 2 + edge_dim + vertex_dim * 2
        self.lin = Linear(head_input_dim, node_dim)
        self.lin_final = Linear(node_dim, 1)

    def forward(self, z_src, z_dst, edge_attr, v_src=None, v_dst=None):
        features = [z_src, z_dst, edge_attr]
        if v_src is not None and self.vertex_dim > 0:
            features.append(v_src)
        if v_dst is not None and self.vertex_dim > 0:
            features.append(v_dst)
        h = torch.cat(features, dim=-1)
        h = self.lin(h).relu()
        return self.lin_final(h)


class TemporalAnomalyTrainer:
    def __init__(
        self,
        memory: TGNMemory,
        gnn: torch.nn.Module,
        edge_head: torch.nn.Module,
        data: TemporalData,
        neighbor_loader: LastNeighborLoader,
        train_loader: TemporalDataLoader,
        val_loader: TemporalDataLoader,
        test_loader: TemporalDataLoader,
        criterion,
        Optimizer: type[torch.optim.Optimizer],
        checkpoint_dir: Path,
        log_dir: Path,
        device: Literal['cpu', 'mps'] = 'cpu',
        vertices: torch.Tensor | None = None,
    ) -> None:
        self.device = device
        self.memory = memory.to(self.device)
        self.gnn = gnn.to(self.device)
        self.edge_head = edge_head.to(self.device)
        self.data = data
        self.vertices = vertices.to(self.device) if vertices is not None else None
        self.neighbor_loader = neighbor_loader
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.criterion = criterion.to(self.device)
        self.optimizer = Optimizer(
            list(self.memory.parameters()) + list(self.gnn.parameters()) + list(self.edge_head.parameters()),
            lr=1e-4,
        )
        self.checkpoint_dir = checkpoint_dir
        self.log_dir = log_dir
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.log_dir.mkdir(parents=True, exist_ok=True)
        self.assoc = torch.empty(data.num_nodes, dtype=torch.long, device=self.device)
        self.epochs_run = 0
        self.best_val_score = float('-inf')
        self.best_threshold = 0.5
        self.history: list[dict[str, float]] = []

    def train_mode(self) -> None:
        self.memory.train()
        self.gnn.train()
        self.edge_head.train()

    def eval_mode(self) -> None:
        self.memory.eval()
        self.gnn.eval()
        self.edge_head.eval()

    def _reset_temporal_state(self) -> None:
        self.memory.reset_state()
        self.neighbor_loader.reset_state()

    def _run_train_batch(self, batch) -> float:
        batch = batch.to(self.device)
        self.optimizer.zero_grad()

        n_id, edge_index, e_id = self.neighbor_loader(batch.n_id)
        self.assoc[n_id] = torch.arange(n_id.size(0), device=self.device)

        z, last_update = self.memory(n_id)
        # Pass vertex features for early fusion into the GNN
        z = self.gnn(
            z,
            last_update,
            edge_index,
            self.data.t[e_id].to(self.device),
            self.data.msg[e_id].to(self.device),
            vertex_features=self.vertices[n_id] if self.vertices is not None else None,
        )

        # Provide vertex features to the edge head as well (multi-scale fusion)
        v_src = self.vertices[batch.src] if self.vertices is not None else None
        v_dst = self.vertices[batch.dst] if self.vertices is not None else None

        logits = self.edge_head(
            z[self.assoc[batch.src]],
            z[self.assoc[batch.dst]],
            batch.msg,
            v_src=v_src,
            v_dst=v_dst,
        )
        loss = self.criterion(logits.squeeze(-1), batch.y.float())

        loss.backward()
        self.optimizer.step()

        # Mutate temporal state only after backprop to avoid in-place autograd version errors.
        self.memory.update_state(batch.src, batch.dst, batch.t, batch.msg)
        self.neighbor_loader.insert(batch.src, batch.dst)
        self.memory.detach()
        return float(loss.item())

    def _train_epoch(self) -> float:
        losses = []
        progress_bar = tqdm(self.train_loader, total=len(self.train_loader), leave=False)
        for batch in progress_bar:
            batch_loss = self._run_train_batch(batch)
            losses.append(batch_loss)
            progress_bar.set_postfix({'loss': f'{np.mean(losses):.4f}'})
        return float(np.mean(losses)) if losses else float('nan')

    @torch.no_grad()
    def _predict_loader(self, loader: TemporalDataLoader) -> tuple[float, np.ndarray, np.ndarray]:
        self.eval_mode()
        losses = []
        y_true_batches = []
        y_score_batches = []

        for batch in loader:
            batch = batch.to(self.device)
            n_id, edge_index, e_id = self.neighbor_loader(batch.n_id)
            self.assoc[n_id] = torch.arange(n_id.size(0), device=self.device)

            z, last_update = self.memory(n_id)
            z = self.gnn(
                z,
                last_update,
                edge_index,
                self.data.t[e_id].to(self.device),
                self.data.msg[e_id].to(self.device),
                vertex_features=self.vertices[n_id] if self.vertices is not None else None,
            )

            v_src = self.vertices[batch.src] if self.vertices is not None else None
            v_dst = self.vertices[batch.dst] if self.vertices is not None else None

            logits = self.edge_head(
                z[self.assoc[batch.src]],
                z[self.assoc[batch.dst]],
                batch.msg,
                v_src=v_src,
                v_dst=v_dst,
            )
            loss = self.criterion(logits.squeeze(-1), batch.y.float())

            losses.append(float(loss.item()))
            y_true_batches.append(batch.y.detach().cpu().to(torch.int64))
            y_score_batches.append(torch.sigmoid(logits.squeeze(-1)).detach().cpu())

        y_true = torch.cat(y_true_batches).numpy() if y_true_batches else np.array([], dtype=np.int64)
        y_score = torch.cat(y_score_batches).numpy() if y_score_batches else np.array([], dtype=np.float32)
        avg_loss = float(np.mean(losses)) if losses else float('nan')
        return avg_loss, y_true, y_score

    def _evaluate_split(self, loader: TemporalDataLoader, threshold: float | None = None) -> tuple[float, dict[str, float], np.ndarray, np.ndarray]:
        avg_loss, y_true, y_score = self._predict_loader(loader)
        calibrated_threshold = calibrate_threshold(y_true, y_score) if threshold is None else float(threshold)
        metrics = compute_metrics(y_true, y_score, calibrated_threshold)
        metrics['loss'] = avg_loss
        return avg_loss, metrics, y_true, y_score

    def evaluate_loader(self, loader: TemporalDataLoader, threshold: float | None = None) -> tuple[float, dict[str, float], np.ndarray, np.ndarray]:
        return self._evaluate_split(loader, threshold=threshold)

    def evaluate_test(self, threshold: float | None = None) -> tuple[float, dict[str, float], np.ndarray, np.ndarray]:
        return self._evaluate_split(self.test_loader, threshold=threshold)

    def _save_checkpoint(self, path: Path, epoch: int, metrics: dict[str, float]) -> None:
        checkpoint = {
            'epoch': epoch,
            'best_val_score': self.best_val_score,
            'best_threshold': self.best_threshold,
            'memory_state_dict': self.memory.state_dict(),
            'gnn_state_dict': self.gnn.state_dict(),
            'edge_head_state_dict': self.edge_head.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'neighbor_loader': self.neighbor_loader,
            'metrics': metrics,
        }
        torch.save(checkpoint, path)

    def load_checkpoint(self, path: Path) -> dict:
        checkpoint = torch.load(path, map_location=self.device, weights_only=False)
        self.memory.load_state_dict(checkpoint['memory_state_dict'])
        self.gnn.load_state_dict(checkpoint['gnn_state_dict'])
        self.edge_head.load_state_dict(checkpoint['edge_head_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.neighbor_loader = checkpoint['neighbor_loader']
        self.best_val_score = float(checkpoint.get('best_val_score', float('-inf')))
        self.best_threshold = float(checkpoint.get('best_threshold', 0.5))
        self.epochs_run = int(checkpoint.get('epoch', 0)) + 1
        return checkpoint

    def fit(self, max_epochs: int) -> list[dict[str, float]]:
        run_id = datetime.now().strftime('%Y%m%d-%H%M%S')
        writer_path = self.log_dir / run_id
        history: list[dict[str, float]] = []

        with SummaryWriter(str(writer_path)) as writer:
            for epoch in range(self.epochs_run, max_epochs):
                self._reset_temporal_state()
                self.train_mode()
                start_time = perf_counter()
                train_loss = self._train_epoch()
                train_time = perf_counter() - start_time

                self.eval_mode()
                val_loss, val_metrics, _, _ = self._evaluate_split(self.val_loader)
                self.best_threshold = val_metrics['threshold']

                epoch_summary = {
                    'epoch': float(epoch),
                    'train_loss': float(train_loss),
                    'val_loss': float(val_loss),
                    'val_pr_auc': float(val_metrics['pr_auc']),
                    'val_roc_auc': float(val_metrics['roc_auc']),
                    'val_f1': float(val_metrics['f1']),
                    'val_precision': float(val_metrics['precision']),
                    'val_recall': float(val_metrics['recall']),
                    'threshold': float(val_metrics['threshold']),
                    'train_time_seconds': float(train_time),
                }
                history.append(epoch_summary)
                self.history.append(epoch_summary)

                writer.add_scalar('Loss/train', train_loss, epoch)
                writer.add_scalar('Loss/validation', val_loss, epoch)
                writer.add_scalar('Metrics/val_pr_auc', val_metrics['pr_auc'], epoch)
                writer.add_scalar('Metrics/val_roc_auc', val_metrics['roc_auc'], epoch)
                writer.add_scalar('Metrics/val_f1', val_metrics['f1'], epoch)
                writer.add_scalar('Threshold/validation', val_metrics['threshold'], epoch)

                latest_path = self.checkpoint_dir / 'latest.pt'
                self._save_checkpoint(latest_path, epoch, val_metrics)

                if val_metrics['pr_auc'] > self.best_val_score:
                    self.best_val_score = val_metrics['pr_auc']
                    best_path = self.checkpoint_dir / 'best.pt'
                    self._save_checkpoint(best_path, epoch, val_metrics)

                print(f'Epoch: {epoch} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Time: {train_time:.2f}s')
                print(f"Validation threshold: {val_metrics['threshold']:.4f} | PR AUC: {val_metrics['pr_auc']:.4f} | ROC AUC: {val_metrics['roc_auc']:.4f} | F1: {val_metrics['f1']:.4f}")

                self.epochs_run += 1

        return history

In [9]:
def set_seed(seed) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.mps.manual_seed(seed)
    torch.use_deterministic_algorithms(True)

In [10]:
set_seed(124)

embedding_dim = memory_dim = time_dim = 64
memory = TGNMemory(
    data.num_nodes,
    data.msg.size(-1),
    memory_dim,
    time_dim,
    message_module=IdentityMessage(data.msg.size(-1), memory_dim, time_dim),
    aggregator_module=MeanAggregator(),
)

# Calculate vertex dimension from preprocessed vertex features.
vertex_dim = vertices.shape[1] if 'vertices' in globals() and vertices is not None else 0
print(f"Vertex dimension: {vertex_dim}")

# Keep the GNN output width aligned with the requested embedding size.
gnn = GraphAttentionEmbedding(
    in_channels=memory_dim,
    out_channels=embedding_dim,
    msg_dim=data.msg.size(-1),
    time_enc=memory.time_enc,
    vertex_dim=vertex_dim,
    num_heads=4,
)
print(f"Node embedding dimension: {gnn.output_dim}")

# Multi-scale fusion: edge head also receives vertex features.
edge_head = EdgeAnomalyHead(
    node_dim=gnn.output_dim,
    edge_dim=data.msg.size(-1),
    vertex_dim=vertex_dim,
)

# Quick forward-shape diagnostic
print("\n--- Forward-shape diagnostic ---")
print(f"vertices.shape: {vertices.shape}")
print(f"edge feature dim: {data.msg.size(-1)}")

# Test forward pass with vertex features.
test_batch_size = 16
test_z = torch.randn(test_batch_size, memory_dim)
test_last_update = torch.zeros(test_batch_size, dtype=torch.long)
test_edge_index = torch.randint(0, test_batch_size, (2, 32))
test_t = torch.zeros(32, dtype=torch.long)
test_msg = torch.randn(32, data.msg.size(-1))
test_vertices = vertices[:test_batch_size]

# Forward through GNN with vertex features.
z_out = gnn(test_z, test_last_update, test_edge_index, test_t, test_msg, vertex_features=test_vertices)
print(f"   - GNN output shape: {z_out.shape} ✓")

# Forward through edge head with vertex features.
test_src_idx = torch.tensor([0, 1, 2, 3])
test_dst_idx = torch.tensor([4, 5, 6, 7])
test_edge_attr = torch.randn(4, data.msg.size(-1))
test_v_src = vertices[test_src_idx]
test_v_dst = vertices[test_dst_idx]

logits = edge_head(z_out[test_src_idx], z_out[test_dst_idx], test_edge_attr, v_src=test_v_src, v_dst=test_v_dst)
print(f"   - Edge head output shape: {logits.shape} ✓")

print("\n✓ Early-fusion architecture successfully integrated (forward shapes OK)!")

Vertex dimension: 12
Node embedding dimension: 64

--- Forward-shape diagnostic ---
vertices.shape: torch.Size([518581, 12])
edge feature dim: 44
   - GNN output shape: torch.Size([16, 64]) ✓
   - Edge head output shape: torch.Size([4, 1]) ✓

✓ Early-fusion architecture successfully integrated (forward shapes OK)!


In [11]:
# Short training run: one epoch to validate the full pipeline end-to-end

def calibrate_threshold(y_true: np.ndarray, y_score: np.ndarray) -> float:
    if np.unique(y_true).size < 2:
        return 0.5
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    if thresholds.size == 0:
        return 0.5
    f1_values = (2 * precision[:-1] * recall[:-1]) / np.clip(precision[:-1] + recall[:-1], 1e-12, None)
    return float(thresholds[int(np.nanargmax(f1_values))])


def compute_metrics(y_true: np.ndarray, y_score: np.ndarray, threshold: float) -> dict[str, float]:
    y_pred = (y_score >= threshold).astype(np.int64)
    metrics = {
        'threshold': float(threshold),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
    }
    if np.unique(y_true).size > 1:
        metrics['roc_auc'] = float(roc_auc_score(y_true, y_score))
        metrics['pr_auc'] = float(average_precision_score(y_true, y_score))
    else:
        metrics['roc_auc'] = float('nan')
        metrics['pr_auc'] = float('nan')
    return metrics

trainer = TemporalAnomalyTrainer(
    memory=memory,
    gnn=gnn,
    edge_head=edge_head,
    data=data,
    neighbor_loader=neighbor_loader,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    criterion=torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([100.0], dtype=torch.float32)),
    Optimizer=AdamW,
    checkpoint_dir=checkpoint_dir,
    log_dir=log_dir,
    device=device,
    vertices=vertices,
)

history = trainer.fit(max_epochs=5)
print(history[-1] if history else {})

/Users/darrellcr/Devs/apple_aiml/c1/tabular/python/.venv/lib/python3.13/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


Epoch: 0 | Train Loss: 6.9226 | Val Loss: 4.6365 | Time: 312.89s
Validation threshold: 0.9608 | PR AUC: 0.0078 | ROC AUC: 0.6575 | F1: 0.0398


Epoch: 1 | Train Loss: 6.4596 | Val Loss: 2.3422 | Time: 321.50s
Validation threshold: 0.8715 | PR AUC: 0.0081 | ROC AUC: 0.7070 | F1: 0.0286


Epoch: 2 | Train Loss: 6.3575 | Val Loss: 2.8269 | Time: 312.03s
Validation threshold: 0.8460 | PR AUC: 0.0088 | ROC AUC: 0.7193 | F1: 0.0344


Epoch: 3 | Train Loss: 2.8486 | Val Loss: 2.4620 | Time: 319.07s
Validation threshold: 0.8049 | PR AUC: 0.0080 | ROC AUC: 0.7238 | F1: 0.0317


Epoch: 4 | Train Loss: 3.6946 | Val Loss: 2.1705 | Time: 313.30s
Validation threshold: 0.7821 | PR AUC: 0.0108 | ROC AUC: 0.7565 | F1: 0.0404
{'epoch': 4.0, 'train_loss': 3.6945820610794957, 'val_loss': 2.1704623632647357, 'val_pr_auc': 0.010762699690746664, 'val_roc_auc': 0.7564572161559779, 'val_f1': 0.04040404040404041, 'val_precision': 0.021642984849910604, 'val_recall': 0.3034300791556728, 'threshold': 0.7820979356765747, 'train_time_seconds': 313.29696608299855}
